# Notebook 02: Entrenamiento LoRA

---
### Objetivo
Hacer fine-tuning con LoRA de `Qwen/Qwen2.5-Coder-1.5B-Instruct` sobre `datos/entrenamiento/train.jsonl` (2,429 pares pregunta–SQL en español, MySQL 8.4).

Se entrenan **dos configuraciones que difieren en un solo factor: el rango `r`** (con $\alpha = 2r$). Todo lo demás (datos, prompt, learning rate, épocas, batch, semilla y hardware: Colab T4) es idéntico.

| | Config 1 | Config 2 |
|---|---|---|
| `r` | 8 | 32 |
| `lora_alpha` | 16 | 64 |
| `target_modules` | q,k,v,o,gate,up,down | q,k,v,o,gate,up,down |
| learning rate | 2e-4 | 2e-4 |
| épocas | 2 | 2 |
| batch efectivo | 16 (2 × 8 acumulación) | 16 (2 × 8 acumulación) |

**Cómo usarlo:** correr el notebook completo con `R = 8`, luego reiniciar el entorno y correrlo otra vez con `R = 32`.

### Hipótesis (escritas ANTES de entrenar)

**H1 — Capacidad vs. tamaño del dataset.** Con `r = 32` el adapter tiene 4× más parámetros entrenables que con `r = 8`. Esperamos que la **loss de entrenamiento** de `r = 32` baje más rápido y termine más baja.

**H2 — Generalización.** Como el dataset es pequeño (~2.4k pares) y los esquemas de la evaluación son distintos a los de entrenamiento, esperamos que la **diferencia en EX sobre la validación propia sea pequeña** (≤ 5 puntos). Si `r = 32` tiene una loss de validación que deja de bajar o sube mientras la de entrenamiento sigue bajando, será señal de sobreajuste y `r = 8` generalizará igual o mejor.

**H3 — Dónde se nota la diferencia.** Si `r = 32` gana, esperamos que la ganancia esté en preguntas **difíciles** (subconsultas, `WITH`, ventanas), que requieren aprender patrones más complejos; en las fáciles ambas deberían ser similares.

**H4 — Costo.** El tiempo de entrenamiento de `r = 32` será solo un poco mayor (≈ +10–20 %), porque el costo lo domina el forward/backward del modelo base, no los adapters.

In [ ]:
# Setup universal
import os, sys, subprocess
from pathlib import Path

# Reduce la fragmentación de memoria de la GPU (debe ir antes de usar CUDA)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# Detectar si estamos en Google Colab
EN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if EN_COLAB:
    import getpass
    REPO = "Albu231311/Deep-Proyecto3"
    NOMBRE = "erickguerra2"
    CORREO = "eagi578@gmail.com"   # el mismo correo ligado a la cuenta de GitHub
    if not os.path.exists("/content/repo"):
        token = getpass.getpass("GitHub token: ")
        subprocess.run(f"git clone https://{token}@github.com/{REPO}.git /content/repo", shell=True, check=True)
    os.chdir("/content/repo")
    subprocess.run("git pull", shell=True, check=True)
    subprocess.run(f'git config user.name "{NOMBRE}"', shell=True)
    subprocess.run(f'git config user.email "{CORREO}"', shell=True)
    subprocess.run("pip install -q -r requirements.txt", shell=True, check=True)
    ROOT_DIR = Path("/content/repo")
else:

    if Path.cwd().name == "notebooks":
        os.chdir(Path.cwd().parent)
    ROOT_DIR = Path.cwd().resolve()

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

print(f" Entorno listo. Directorio de trabajo: {os.getcwd()} | En Colab: {EN_COLAB}")

In [ ]:
# Los adapters de cada configuración se guardan en el repo y se suben a GitHub al final (sección 7).
DIR_ADAPTERS = ROOT_DIR / "adapters_experimentos"
DIR_ADAPTERS.mkdir(parents=True, exist_ok=True)

# Copia extra en Google Drive. Solo funciona en colab.research.google.com;
# desde la extensión de VS Code dejarlo en False.
USAR_DRIVE = False
DIR_DRIVE = None
if USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DIR_DRIVE = Path("/content/drive/MyDrive/Proyecto3_adapters")
    DIR_DRIVE.mkdir(parents=True, exist_ok=True)
print("Adapters se guardan en:", DIR_ADAPTERS, "| Drive:", DIR_DRIVE)

## 1. Configuración del experimento

**Único factor que cambia entre corridas: `R`.**

In [ ]:
R = 8            # <-- Config 1: 8   |   Config 2: 32
ALPHA = 2 * R

MODELO_BASE = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
SEMILLA = 2026
LR = 2e-4
EPOCAS = 2
# Batch 2 x 8: con 4 x 4 los logits (4 x 2048 tokens x 151,936 de vocabulario en fp32,
# ~4.6 GB) no caben en la T4. El batch efectivo sigue siendo 16.
BATCH = 2
ACUMULACION = 8          # batch efectivo = 16
MAX_LEN = 2048           # ejemplos más largos se descartan (se reporta cuántos)
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

NOMBRE_RUN = f"r{R}"
import torch
print(f"Config {NOMBRE_RUN}: r={R}, alpha={ALPHA} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NINGUNA'}")

## 2. Datos

Se separa ~5 % de las **bases** de entrenamiento (partición por `db_id`, no por pregunta) solo para graficar la loss de validación durante el entrenamiento. La validación propia de 40 preguntas (`datos/validacion/`) no se usa aquí; se evalúa por ejecución en el notebook 03.

In [ ]:
import json, random
from collections import Counter

with open("datos/entrenamiento/train.jsonl", encoding="utf-8") as f:
    datos = [json.loads(l) for l in f]

bases = sorted({d["db_id"] for d in datos})
rng = random.Random(SEMILLA)
rng.shuffle(bases)
bases_eval = set(bases[: max(3, len(bases) // 20)])

train_raw = [d for d in datos if d["db_id"] not in bases_eval]
eval_raw = [d for d in datos if d["db_id"] in bases_eval]
print(f"Total: {len(datos)} | train: {len(train_raw)} | eval (loss): {len(eval_raw)} | bases eval: {sorted(bases_eval)}")
print("Niveles train:", Counter(d["nivel"] for d in train_raw))

## 3. Tokenización con el prompt oficial y enmascarado del prompt

Se usa **exactamente** el mismo prompt que en `solucion.py` (`src/prompt.py`). Los tokens del prompt (system + esquema + pregunta) llevan `label = -100`, así la loss se calcula **solo sobre la consulta SQL** (y el token de fin `<|im_end|>`, para que aprenda a detenerse).

In [ ]:
from transformers import AutoTokenizer
from src.prompt import construir_mensajes

tokenizer = AutoTokenizer.from_pretrained(MODELO_BASE)

def tokenizar(ej):
    mensajes = construir_mensajes(ej["esquema"], ej["pregunta"])
    prompt = tokenizer.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=True)
    completo = tokenizer.apply_chat_template(
        mensajes + [{"role": "assistant", "content": ej["sql"]}], tokenize=False
    )
    assert completo.startswith(prompt)
    ids_prompt = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    ids = tokenizer(completo, add_special_tokens=False)["input_ids"]
    labels = [-100] * len(ids_prompt) + ids[len(ids_prompt):]
    return {"input_ids": ids, "labels": labels}

def preparar(lista):
    salida, largos = [], []
    for ej in lista:
        t = tokenizar(ej)
        largos.append(len(t["input_ids"]))
        if len(t["input_ids"]) <= MAX_LEN:
            salida.append(t)
    return salida, largos

train_tok, largos = preparar(train_raw)
eval_tok, _ = preparar(eval_raw)

import numpy as np
print(f"Tokens por ejemplo: media={np.mean(largos):.0f}, p95={np.percentile(largos, 95):.0f}, max={max(largos)}")
print(f"Descartados por > {MAX_LEN} tokens: {len(train_raw) - len(train_tok)}")
print(f"Ejemplos finales -> train: {len(train_tok)} | eval: {len(eval_tok)}")

# Verificación visual del enmascarado
ej = train_tok[0]
print("\n--- Parte que SÍ entra en la loss ---")
print(tokenizer.decode([t for t, l in zip(ej["input_ids"], ej["labels"]) if l != -100]))

## 4. Modelo base + LoRA

El modelo base se carga en `float16` (T4 no soporta bf16). Los pesos LoRA se pasan a `float32` para que el entrenamiento con precisión mixta (`fp16=True`) sea estable.

In [ ]:
from transformers import AutoModelForCausalLM, set_seed
from peft import LoraConfig, get_peft_model
from src.compat import parchear_torchao

parchear_torchao()   # Colab trae un torchao viejo que rompe peft (ver src/compat.py)
set_seed(SEMILLA)

modelo = AutoModelForCausalLM.from_pretrained(MODELO_BASE, dtype=torch.float16).to("cuda")
modelo.config.use_cache = False
modelo.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
modelo.enable_input_require_grads()

lora_config = LoraConfig(
    r=R,
    lora_alpha=ALPHA,
    lora_dropout=0.05,
    target_modules=TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)
modelo = get_peft_model(modelo, lora_config)

for p in modelo.parameters():
    if p.requires_grad:
        p.data = p.data.float()

modelo.print_trainable_parameters()
entrenables = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
totales = sum(p.numel() for p in modelo.parameters())

## 5. Entrenamiento

In [ ]:
import time, inspect
from transformers import Trainer, TrainingArguments

def collate(batch):
    # Padding a la derecha; las posiciones de padding no cuentan en la loss
    largo = max(len(b["input_ids"]) for b in batch)
    pad = tokenizer.pad_token_id
    input_ids = [b["input_ids"] + [pad] * (largo - len(b["input_ids"])) for b in batch]
    labels = [b["labels"] + [-100] * (largo - len(b["labels"])) for b in batch]
    mask = [[1] * len(b["input_ids"]) + [0] * (largo - len(b["input_ids"])) for b in batch]
    return {
        "input_ids": torch.tensor(input_ids),
        "labels": torch.tensor(labels),
        "attention_mask": torch.tensor(mask),
    }

# Agrupar ejemplos de largo parecido (menos padding = más rápido).
# transformers v5 renombró group_by_length -> train_sampling_strategy
if "train_sampling_strategy" in inspect.signature(TrainingArguments).parameters:
    agrupar = {"train_sampling_strategy": "group_by_length"}
else:
    agrupar = {"group_by_length": True}

args = TrainingArguments(
    output_dir=str(ROOT_DIR / "checkpoints" / NOMBRE_RUN),
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=ACUMULACION,
    num_train_epochs=EPOCAS,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_steps=10,         # ~3 % de los ~295 pasos

    fp16=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="no",
    report_to="none",
    seed=SEMILLA,
    remove_unused_columns=False,
    **agrupar,
)

trainer = Trainer(
    model=modelo,
    args=args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    data_collator=collate,
)

inicio = time.time()
trainer.train()
tiempo_min = (time.time() - inicio) / 60
print(f"\nTiempo de entrenamiento ({NOMBRE_RUN}): {tiempo_min:.1f} min")

## 6. Guardar adapter, métricas y curvas de loss

In [ ]:
import matplotlib.pyplot as plt

# Adapter (adapter_config.json + adapter_model.safetensors) en float16:
# r=32 en float32 pesaría ~148 MB (> 100 MB permitidos); en float16 ~74 MB
modelo = modelo.to(torch.float16)
dir_adapter = DIR_ADAPTERS / f"adapter_{NOMBRE_RUN}"
modelo.save_pretrained(dir_adapter)
if DIR_DRIVE:
    import shutil
    shutil.copytree(dir_adapter, DIR_DRIVE / dir_adapter.name, dirs_exist_ok=True)
tam_mb = sum(f.stat().st_size for f in dir_adapter.glob("*")) / 1e6
print(f"Adapter guardado en {dir_adapter} ({tam_mb:.1f} MB)")

# Historial de loss
historial = trainer.state.log_history
train_pts = [(h["step"], h["loss"]) for h in historial if "loss" in h]
eval_pts = [(h["step"], h["eval_loss"]) for h in historial if "eval_loss" in h]

dir_curvas = ROOT_DIR / "resultados" / "curvas_loss"
dir_curvas.mkdir(parents=True, exist_ok=True)

metricas = {
    "config": NOMBRE_RUN,
    "r": R,
    "lora_alpha": ALPHA,
    "target_modules": TARGET_MODULES,
    "learning_rate": LR,
    "epocas": EPOCAS,
    "batch_efectivo": BATCH * ACUMULACION,
    "parametros_entrenables": entrenables,
    "parametros_totales": totales,
    "porcentaje_entrenable": 100 * entrenables / totales,
    "tiempo_entrenamiento_min": round(tiempo_min, 2),
    "gpu": torch.cuda.get_device_name(0),
    "ejemplos_train": len(train_tok),
    "ejemplos_eval": len(eval_tok),
    "train_loss_final": train_pts[-1][1],
    "eval_loss_final": eval_pts[-1][1] if eval_pts else None,
    "tamano_adapter_mb": round(tam_mb, 1),
    "historial": historial,
}
with open(dir_curvas / f"metricas_{NOMBRE_RUN}.json", "w", encoding="utf-8") as f:
    json.dump(metricas, f, indent=2, ensure_ascii=False)

plt.figure(figsize=(8, 4.5))
plt.plot(*zip(*train_pts), label="train")
if eval_pts:
    plt.plot(*zip(*eval_pts), "o-", label="validación (bases retenidas)")
plt.xlabel("paso")
plt.ylabel("loss")
plt.title(f"Curva de loss — LoRA r={R}, alpha={ALPHA}")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(dir_curvas / f"loss_{NOMBRE_RUN}.png", dpi=150)
plt.show()

print({k: v for k, v in metricas.items() if k != "historial"})

## 7. Subir adapter, curvas y métricas a GitHub

La máquina de Colab se borra al cerrar la sesión, así que los resultados se suben al repo desde aquí (con el usuario configurado en el setup). Después, en tu PC: `git pull`.

In [ ]:
import re

def subir(rutas, mensaje):
    # git add + commit + push desde Colab; oculta el token en lo que se imprime
    def git(cmd):
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=ROOT_DIR)
        salida = re.sub(r"https://[^@\s]+@", "https://***@", r.stdout + r.stderr).strip()
        if salida:
            print(salida)
        return r.returncode
    rutas = [str(r) for r in rutas if (ROOT_DIR / r).exists()]
    git("git add " + " ".join(f'"{r}"' for r in rutas))
    if git(f'git commit -m "{mensaje}"') != 0:
        print("Nada nuevo que subir.")
        return
    git("git pull --rebase")
    print("Push OK" if git("git push") == 0 else "ERROR en el push")

subir(
    [
        f"adapters_experimentos/adapter_{NOMBRE_RUN}",
        f"resultados/curvas_loss/loss_{NOMBRE_RUN}.png",
        f"resultados/curvas_loss/metricas_{NOMBRE_RUN}.json",
    ],
    f"feat(entrenamiento): adapter LoRA {NOMBRE_RUN} y curvas de loss",
)

## 8. Comparación de ambas configuraciones

Correr esta sección cuando ya existan `metricas_r8.json` y `metricas_r32.json`.

In [ ]:
import pandas as pd

filas, plt_ok = [], False
plt.figure(figsize=(8, 4.5))
for nombre in ["r8", "r32"]:
    p = ROOT_DIR / "resultados" / "curvas_loss" / f"metricas_{nombre}.json"
    if not p.exists():
        continue
    m = json.load(open(p, encoding="utf-8"))
    filas.append({k: m[k] for k in ["config", "parametros_entrenables", "porcentaje_entrenable",
                                    "tiempo_entrenamiento_min", "train_loss_final", "eval_loss_final",
                                    "tamano_adapter_mb"]})
    h = m["historial"]
    plt.plot([x["step"] for x in h if "loss" in x], [x["loss"] for x in h if "loss" in x], label=f"{nombre} train")
    plt.plot([x["step"] for x in h if "eval_loss" in x], [x["eval_loss"] for x in h if "eval_loss" in x],
             "o--", label=f"{nombre} validación")
    plt_ok = True

if plt_ok:
    plt.xlabel("paso"); plt.ylabel("loss"); plt.title("r=8 vs r=32")
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
    plt.savefig(ROOT_DIR / "resultados" / "curvas_loss" / "comparacion_r8_r32.png", dpi=150)
    plt.show()
    subir(["resultados/curvas_loss/comparacion_r8_r32.png"],
          "docs(entrenamiento): comparación de curvas de loss r8 vs r32")
pd.DataFrame(filas)

## 9. Copiar el mejor adapter a `adapter/`

El mejor se elige por **EX en la validación propia** (notebook 03), no por la loss.

In [ ]:
import shutil

MEJOR = "r8"   # <-- actualizar después de evaluar en el notebook 03

origen = ROOT_DIR / "adapters_experimentos" / f"adapter_{MEJOR}"
destino = ROOT_DIR / "adapter"
destino.mkdir(exist_ok=True)
for nombre in ["adapter_config.json", "adapter_model.safetensors"]:
    shutil.copy(origen / nombre, destino / nombre)
print(sorted(p.name for p in destino.iterdir()))
subir(["adapter/adapter_config.json", "adapter/adapter_model.safetensors"],
      f"feat(entrenamiento): adapter final ({MEJOR}) en adapter/")